### BulkFormer feature extraction

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

In [16]:
import math
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader,random_split
from utils.BulkFormer import BulkFormer
from model.config import model_params

In [17]:
# Configuration
device = 'cuda'
graph_path = 'data/G_tcga.pt'
weights_path = 'data/G_tcga_weight.pt'
gene_emb_path = 'data/esm2_feature_concat.pt'

In [18]:
# Initialize the BulkFormer model with preloaded graph structure and gene embeddings.
# Use (edge_index, edge_weight) fallback when torch-sparse is unavailable (e.g. CUDA mismatch).
graph_obj = torch.load(graph_path, map_location=device, weights_only=False)
weights = torch.load(weights_path, map_location=device, weights_only=False)
row, col = graph_obj[1], graph_obj[0]
if not torch.is_tensor(weights):
    weights = torch.as_tensor(weights, device=device)
try:
    from torch_geometric.typing import SparseTensor
    graph = SparseTensor(row=row, col=col, value=weights).t().to(device)
except Exception:
    edge_index = torch.stack([col, row], dim=0).long().to(device)
    edge_weight = weights.to(device)
    graph = (edge_index, edge_weight)
gene_emb = torch.load(gene_emb_path, map_location=device, weights_only=False)
model_params['graph'] = graph
model_params['gene_emb'] = gene_emb
model = BulkFormer(**model_params).to(device)

In [20]:
# Load the pretrained BulkFormer model checkpoint for inference or fine-tuning.
ckpt_model = torch.load('model/BulkFormer_147M.pt',weights_only=False)

new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith("module.") else key
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict)

<All keys matched successfully>

In [21]:
def normalize_data(X_df, gene_length_dict):
    """
    Normalize RNA-seq count data to log-transformed TPM values.

    Parameters
    ----------
    X_df : pandas.DataFrame
        A gene expression matrix where rows represent samples and columns represent genes.
        Each entry contains the raw read count of a gene in a given sample.

    gene_length_dict : dict
        A dictionary mapping gene identifiers (Ensembl gene IDs) to gene lengths (in base pairs).

    Returns
    -------
    log_tpm_df : pandas.DataFrame
        A DataFrame of the same shape as `X_df`, containing log-transformed TPM values
        (i.e., log(TPM + 1)) for each gene in each sample.

    Description
    -----------
    This function converts raw RNA-seq count data to transcripts per million (TPM) values by
    normalizing for gene length and sample-specific total expression. Gene lengths are provided
    via `gene_length_dict`, and genes not present in the dictionary are assigned a default
    length of 1,000 bp (equivalent to no correction). The resulting TPM values are subsequently
    log-transformed using the natural logarithm (log1p). This normalization procedure accounts
    for both gene length and sequencing depth, facilitating cross-sample and cross-gene comparisons.
    """
    gene_names = X_df.columns
    gene_lengths_kb = np.array([gene_length_dict.get(gene, 1000) / 1000  for gene in gene_names])
    counts_matirx = X_df.values
    rate = counts_matirx / gene_lengths_kb
    sum_per_sample = rate.sum(axis=1)
    sum_per_sample[sum_per_sample == 0] = 1e-6  
    sum_per_sample = sum_per_sample.reshape(-1, 1)
    tpm = rate / sum_per_sample * 1e6
    log_tpm = np.log1p(tpm)
    log_tpm_df = pd.DataFrame(log_tpm,index=X_df.index, columns=X_df.columns)
    return log_tpm_df

In [22]:
def main_gene_selection(X_df, gene_list):
    """
    Aligns a gene expression matrix to a predefined gene list by adding placeholder values
    for missing genes and generating a binary mask indicating imputed entries.

    Parameters
    ----------
    X_df : pandas.DataFrame
        A gene expression matrix with rows representing samples and columns representing genes.
        The entries are typically log-transformed or normalized expression values.

    gene_list : list of str
        A predefined list of gene identifiers (Ensembl Gene IDs) to be retained
        in the final matrix. This list defines the desired gene space for downstream analyses.

    Returns
    -------
    X_df : pandas.DataFrame
        A gene expression matrix aligned to `gene_list`, with missing genes filled with a constant
        placeholder value (−10) and columns ordered accordingly.

    to_fill_columns : list of str
        A list of genes from `gene_list` that were not present in the original `X_df`
        and were therefore added with placeholder values.

    var : pandas.DataFrame
        A DataFrame with one row per gene, containing a binary column `'mask'` indicating
        whether a gene was imputed (1) or originally present (0). This can be used for masking
        in training or evaluation of models that distinguish observed and imputed entries.

    Notes
    -----
    This function ensures that all samples share a consistent gene space, which is essential
    for tasks such as model training, cross-dataset integration, or visualization. Placeholder
    values (−10) are used to maintain matrix shape while avoiding unintended bias in downstream
    statistical analyses or machine learning models.
    """
    to_fill_columns = list(set(gene_list) - set(X_df.columns))

    padding_df = pd.DataFrame(np.full((X_df.shape[0], len(to_fill_columns)), -10), 
                            columns=to_fill_columns, 
                            index=X_df.index)

    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var

In [41]:
def extract_feature(expr_array, 
                    output_feature_type,
                    aggregate_type,
                    device,
                    batch_size,
                    mask_prob = 0.1,
                    interested_gene_idx = None,
                    output_expr = False,
                    esm2_emb = None):
    
    """
    Extracts sample-level or gene-level feature representations from input expression profiles
    using a pre-trained deep learning model.

    Parameters
    ----------
    expr_array : np.ndarray
        A NumPy array of shape [N_samples, N_genes] representing gene expression profiles
        (e.g., log-transformed TPM values).

    mask_prob : float, optional
        Masking probability used during feature extraction to simulate missing genes.
        This parameter controls the fraction of randomly masked genes during inference,
        enabling robustness to incomplete gene sets relative to the pretrained vocabulary.

    interested_gene_idx : list or np.ndarray
        Indices of your interested genes used for sample-level embedding aggregation.

    feature_type : str
        Specifies the type of feature to extract. Options:
            - 'sample_level': aggregate gene embeddings to a single sample-level vector.
            - 'gene_level': retain per-gene embeddings for downstream fusion with external embeddings (ESM2).

    aggregate_type : str
        Aggregation method used when `feature_type='sample_level'`. Options include:
            - 'max': use maximum value across selected genes.
            - 'mean': use average value.
            - 'median': use median value.
            - 'all': combine all three strategies by summation.

    device : torch.device
        Computation device (e.g., 'cuda' or 'cpu') for model inference.

    batch_size : int
        Number of samples per batch during feature extraction.

    output_expr : bool, optional
        If True, return predicted gene expression values instead of extracted embeddings.

    esm2_emb : torch.Tensor, optional
        Precomputed ESM2 embeddings for all genes, used in gene-level feature concatenation.
        Required if `feature_type='gene_level'`.

    Returns
    -------
    result_emb : torch.Tensor
        The extracted feature representations:
            - [N_samples, D] for sample-level features.
            - [N_samples, N_genes, D_concat] for gene-level features.

    or (if `output_expr=True`)
    expr_predictions : np.ndarray
        Model-predicted expression profiles.
    """

    # Convert NumPy array to PyTorch tensor and move to target device
    expr_tensor = torch.tensor(expr_array, dtype=torch.float32, device=device)

    # Create dataset and dataloader for batched inference
    mydataset = TensorDataset(expr_tensor)
    myloader = DataLoader(mydataset, batch_size=batch_size, shuffle=False)

    # Switch model to evaluation mode
    model.eval()

    all_emb_list = []
    all_pred_expr_list = []

    # Use inference mode with automatic mixed precision on GPU
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=True):

        # ----------------------------------------------------------
        # 1. SAMPLE-LEVEL FEATURE EXTRACTION
        # ----------------------------------------------------------
        if output_feature_type == 'sample_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)

                # Obtain per-gene embeddings from the model
                gene_emb = model(X, mask_prob=mask_prob, output_expr=output_expr)
                gene_emb = gene_emb.detach().cpu().numpy()

                # Select interested genes if user provided gene indices
                if interested_gene_idx:
                    gene_emb = gene_emb[:, interested_gene_idx, :]

                # Perform aggregation to obtain sample-level embeddings
                if aggregate_type == 'max':
                    final_emb = np.max(gene_emb, axis=1)
                elif aggregate_type == 'mean':
                    final_emb = np.mean(gene_emb, axis=1)
                elif aggregate_type == 'median':
                    final_emb = np.median(gene_emb, axis=1)
                elif aggregate_type == 'all':
                    max_emb = np.max(gene_emb, axis=1)
                    mean_emb = np.mean(gene_emb, axis=1)
                    median_emb = np.median(gene_emb, axis=1)
                    final_emb = max_emb + mean_emb + median_emb

                all_emb_list.append(final_emb)

                # Memory cleanup for large batch inference
                del gene_emb
                torch.cuda.empty_cache()

            # Stack all embeddings across batches
            result_emb = np.vstack(all_emb_list)
            result_emb = torch.tensor(result_emb, device='cpu', dtype=torch.float32)

        # ----------------------------------------------------------
        # 2. GENE-LEVEL FEATURE EXTRACTION (OPTIONALLY CONCAT WITH ESM2)
        # ----------------------------------------------------------
        elif output_feature_type == 'gene_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)

                gene_emb = model(X, mask_prob=mask_prob, output_expr=output_expr)
                gene_emb = gene_emb.detach().cpu().numpy()
                all_emb_list.append(gene_emb)

                del gene_emb
                torch.cuda.empty_cache()

            # Stack to [N_samples, N_genes, D]
            all_emb = np.vstack(all_emb_list)
            all_emb_tensor = torch.tensor(all_emb, device='cpu', dtype=torch.float32)

            # Expand ESM2 embeddings to align with batch dimension: [N_samples, N_genes, D_esm2]
            # Move to CPU to match all_emb_tensor (avoids device mismatch in torch.cat)
            esm2_emb_expanded = esm2_emb.unsqueeze(0).expand(all_emb_tensor.shape[0], -1, -1).cpu()

            # Concatenate model embeddings with ESM2 embeddings
            result_emb = torch.cat([all_emb_tensor, esm2_emb_expanded], dim=-1)

        # ----------------------------------------------------------
        # 3. EXPRESSION-LEVEL OUTPUT (MODEL PREDICTS EXPRESSION)
        # ----------------------------------------------------------
        elif output_feature_type == 'expr_level':
            output_expr = True
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)

                pred_expr = model(X, mask_prob=mask_prob, output_expr=output_expr)
                all_pred_expr_list.append(pred_expr.detach().cpu().numpy())

    # ----------------------------------------------------------
    # Final return logic
    # ----------------------------------------------------------
    if output_expr:
        return np.vstack(all_pred_expr_list)
    else:
        return result_emb

In [42]:
# Load demo normalized data (log-transformed TPM)
log_tpm_df = pd.read_csv('data/demo_normalized_data.csv')

In [43]:
# Load demo count data (raw count)
count_df = pd.read_csv('data/demo_count_data.csv')

In [44]:
# Convert raw counts to normalized expression values (log-transformed TPM)
gene_length_df = pd.read_csv('data/gene_length_df.csv')
gene_length_dict = gene_length_df.set_index('ensg_id')['length'].to_dict()
log_tpm_df = normalize_data(X_df=count_df, gene_length_dict=gene_length_dict)

In [45]:
bulkformer_gene_info = pd.read_csv('data/bulkformer_gene_info.csv')
bulkformer_gene_list = bulkformer_gene_info['ensg_id'].to_list()

In [46]:
# Align expression data to a predefined gene list with placeholder imputation for missing genes.
input_df , to_fill_columns, var= main_gene_selection(X_df=log_tpm_df,gene_list=bulkformer_gene_list)

In [47]:
# Compute the gene-missing rate (masking probability) for the input data.
# This measures the proportion of genes in the sample that are absent from the pretrained vocabulary of 200,10 genes. 
mask_prob = len(list(var[var['mask'] == 1].index)) / var.shape[0]
print(f"mask prob: {mask_prob}")

mask prob: 0.0


In [48]:
# Load the list of genes of interest (each entry must be an Ensembl gene ID
# and must exist in the BulkFormer vocabulary). These genes will be used to
# aggregate their corresponding embeddings into a sample-level representation.
interested_gene_idx = torch.load('data/interested_gene_list.pt',weights_only=False)

In [49]:
# Extract sample-level embedding
res1 = extract_feature(
    expr_array= input_df.values[:16],
    interested_gene_idx=interested_gene_idx,
    output_feature_type='sample_level',
    aggregate_type='mean',
    device=device,
    batch_size=4,
    output_expr=False,
    mask_prob=mask_prob,
    esm2_emb=model_params['gene_emb'])

100%|██████████| 4/4 [00:02<00:00,  1.35it/s]


In [50]:
res1.shape

torch.Size([16, 643])

In [51]:
res1.shape

torch.Size([16, 643])

In [52]:
# Extract gene-level embedding
res2 = extract_feature(
    expr_array= input_df.values[:16],
    interested_gene_idx=interested_gene_idx,
    output_feature_type='gene_level',
    aggregate_type='mean', 
    device=device,
    batch_size=4,
    output_expr=False,
    mask_prob=mask_prob,
    esm2_emb=model_params['gene_emb'])

100%|██████████| 4/4 [00:02<00:00,  1.43it/s]


In [53]:
res2.shape

torch.Size([16, 20010, 1923])

In [54]:
res2

tensor([[[ 1.5307, -0.4205, -0.3943,  ..., -0.0972, -0.1156, -0.0694],
         [-0.3059, -1.1357, -0.3093,  ..., -0.0929, -0.0102,  0.0749],
         [-0.9912,  0.6813, -1.1281,  ..., -0.1507, -0.0174,  0.1455],
         ...,
         [-1.0466,  0.1795, -0.9826,  ..., -0.0316,  0.0078,  0.0943],
         [-0.7377, -1.4268, -0.6383,  ..., -0.0891, -0.0469,  0.1897],
         [-0.6179, -0.8940, -0.3646,  ..., -0.0528, -0.0946,  0.0670]],

        [[ 0.8724,  0.2303,  0.3746,  ..., -0.0972, -0.1156, -0.0694],
         [-0.0407, -1.3911, -0.3296,  ..., -0.0929, -0.0102,  0.0749],
         [-0.1016, -0.0219, -0.4198,  ..., -0.1507, -0.0174,  0.1455],
         ...,
         [-0.6696,  0.4092, -0.5654,  ..., -0.0316,  0.0078,  0.0943],
         [-0.5360, -1.4748, -0.6450,  ..., -0.0891, -0.0469,  0.1897],
         [-0.5379, -1.0071, -0.4270,  ..., -0.0528, -0.0946,  0.0670]],

        [[ 2.2212,  0.1969, -0.0200,  ..., -0.0972, -0.1156, -0.0694],
         [-0.2022, -1.4761, -0.5132,  ..., -0

In [55]:
# Extract expression values
res3 = extract_feature(
    expr_array= input_df.values[:16],
    interested_gene_idx=interested_gene_idx,
    output_feature_type='expr_level',
    aggregate_type='mean', 
    device=device,
    batch_size=4,
    output_expr=True,
    mask_prob=mask_prob,
    esm2_emb=model_params['gene_emb'])

100%|██████████| 4/4 [00:02<00:00,  1.80it/s]


In [56]:
res3.shape

(16, 20010)

In [57]:
res3

array([[2.9804688 , 0.        , 3.3183594 , ..., 2.9101562 , 0.        ,
        0.        ],
       [2.6171875 , 0.        , 4.3515625 , ..., 3.6816406 , 0.        ,
        0.        ],
       [1.9970703 , 0.        , 3.3964844 , ..., 1.9121094 , 0.        ,
        0.        ],
       ...,
       [0.7480469 , 0.        , 3.0351562 , ..., 2.203125  , 0.        ,
        0.        ],
       [0.94433594, 0.        , 3.5644531 , ..., 3.9394531 , 0.        ,
        0.        ],
       [3.2050781 , 0.64746094, 3.4414062 , ..., 3.4941406 , 0.        ,
        0.        ]], dtype=float32)